# Intyoza: training Intyoza's own answer model

**Notebook:** 4 of 5. Trains the model that reads the trusted sources and writes Intyoza's answers. It is **Intyoza's own model**: it runs on Intyoza's machine, and no message from a young person is sent to an outside AI company.

## The model

| | |
|---|---|
| Base model | **AfriqueQwen3.5-4B-Instruct** (McGill NLP, AfriqueLLM): Qwen3.5 adapted to 20 African languages including Kinyarwanda. Licence **CC-BY-4.0**: a fine-tuned version may be used and shared, with credit to McGill NLP |
| Method | **QLoRA**: the base model is loaded in 4-bit and small extra layers (LoRA, about 1% of the weights) are trained. This fits a free Colab T4 |
| Result | `models/intyoza_writer` (about 100 MB): Intyoza's fine-tuning, loaded on top of the base model |

## What it learns (from Intyoza's own data, training split only)

| Task | Input | It must answer |
|---|---|---|
| **Talk** | a turn of a whole conversation (history, the young person's profile, sources: approved answers + document passages) | one of several good replies written for that situation (tab **Conversations**, Approved rows): short unless they ask for more, listening first, ending with a question. The facts come from the approved answers; the words differ between young people |
| **Crisis** | the crisis turns of those conversations, with the approved help texts | a warm reply for that person that always gives the approved safety numbers exactly |
| **Not covered** | a question + sources that do not answer it | `NOT_COVERED` (never guess) |
| **Check** | a question + sources + an answer | *is this the answer to THIS question? are its facts in the sources?* Wrong answers are the answers of the **closest other questions** (the "nearest answer" mistake) and answers with a **changed number** |
| **Understand** | a message (sometimes after another exchange) or an off-topic message | the full question in Kinyarwanda and English, search words, SRH / wellbeing / off topic |

The test split (ways of asking that were **never** used for training) measures the base model **before** and **after** fine-tuning, on the same examples.

## Steps

1. Setup
2. Load Intyoza (matcher and data)
3. Trusted documents and DEMO data
4. Load the base model
5. Kinyarwanda questions for the demo data
6. Training examples
7. Base model on the test examples (before)
8. Fine-tuning
9. Fine-tuned model on the test examples (after)
10. Save

**New data:** edit `Intyoza_Dataset.xlsx` (Questions and Phrasings sheets) and upload it as a new version of the Kaggle dataset `intyoza-data` (new PDFs go in its `documents/` folder). This notebook reads it automatically.

**Kaggle inputs:** `intyoza` (code + matcher), `intyoza-code` (latest code), `intyoza-data` (your data), and the last training notebook's output (`intyoza_outputs`). **Requirements:** run `1_Train.ipynb` first. *Runtime > Change runtime type > T4 GPU*. On Kaggle (Save & Run All) the whole notebook takes about 4 to 5 hours; training stops by itself after 5 hours. If Colab disconnects, run the notebook again: training continues from the last checkpoint (saved every 50 steps on Drive).

## 1. Setup

In [ ]:
import os, sys, glob, importlib, warnings, json
warnings.filterwarnings("ignore")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less wasted GPU memory
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working") and not IN_COLAB
ONLINE = IN_COLAB or IN_KAGGLE        # Colab or Kaggle: models are downloaded from the internet
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    hits = glob.glob("/content/drive/MyDrive/**/Intyoza/intyoza_core.py", recursive=True)
    assert hits, "Upload the Intyoza folder to your Google Drive first."
    os.chdir(os.path.dirname(hits[0]))
    !pip -q install -U sentence-transformers accelerate pypdf peft bitsandbytes requests
elif IN_KAGGLE:      # Kaggle: code, trained results, NEW DATA (dataset intyoza-data) and secrets in one call
    ks = []
    try:   # latest code straight from GitHub (Kaggle secrets INTYOZA_REPO, INTYOZA_BRANCH, GITHUB_TOKEN): no zip uploads
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient()
        def _get(name):
            try:
                return (_s.get_secret(name) or "").strip()
            except Exception:
                return ""
        _repo, _branch, _tok = _get("INTYOZA_REPO"), _get("INTYOZA_BRANCH") or "main", _get("GITHUB_TOKEN")
        if _repo:
            import subprocess, shutil
            shutil.rmtree("/tmp/intyoza_repo", ignore_errors=True)
            _url = f"https://{'x-access-token:' + _tok + '@' if _tok else ''}github.com/{_repo}.git"
            _r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", _branch, _url, "/tmp/intyoza_repo"],
                                capture_output=True, text=True)
            ks = glob.glob("/tmp/intyoza_repo/ai/intyoza_answer_model/kaggle_setup.py")
            if not ks:
                print("Code from GitHub failed:", (_r.stderr.replace(_tok, "***") if _tok else _r.stderr)[-200:])
    except ImportError:
        pass
    ks = ks or glob.glob("/kaggle/input/**/kaggle_setup.py", recursive=True)
    assert ks, "Set the Kaggle secret INTYOZA_REPO (or add the dataset intyoza-code)."
    sys.path.insert(0, os.path.dirname(ks[0]))
    import kaggle_setup
    kaggle_setup.setup()
    !pip -q install -U sentence-transformers accelerate pypdf peft bitsandbytes requests
else:
    assert os.path.exists("intyoza_core.py"), "Open this notebook from inside the Intyoza folder."
import pandas as pd, torch
USE_DEMO_DATA = True        # demo data (public SRH data, step 3). False when Intyoza goes live
FOLDER = os.getcwd()
MODELS = os.path.join(FOLDER, "models")
sys.path.insert(0, FOLDER)
pd.set_option("display.max_colwidth", 300)
import preprocess, intyoza_core, doc_engine, chatbot, answer_engine, get_documents, writer_training
for m in (preprocess, intyoza_core, doc_engine, chatbot, answer_engine, get_documents, writer_training):
    importlib.reload(m)
assert os.path.exists(os.path.join(MODELS, "intyoza_matcher")), "Train the matcher first with 1_Train.ipynb."
assert torch.cuda.is_available(), "Select a GPU: Runtime > Change runtime type > T4 GPU."
print("Folder:", FOLDER, "| GPU:", torch.cuda.get_device_name(0))

## 2. Load Intyoza (matcher and data)

In [ ]:
bot = intyoza_core.load_bot(MODELS, FOLDER, use_documents=False, online=ONLINE)
data, qa, ex = intyoza_core.load_all(FOLDER, verbose=False)
print(f"{len(bot.qa)} approved answers | phrasings: {sum(e['split'] == 'train' for e in ex)} train, {sum(e['split'] == 'test' for e in ex)} test")

## 3. Trusted documents and DEMO data

**Documents:** the official Rwandan documents listed in `documents/SOURCES.json` (Ministry of Health / RBC FP guidelines 2022, FP/ASRH strategic plan 2018-2024, MNCH strategic plan 2018-2024, MIGEPROF GBV policy 2011, Ministerial Order on abortion conditions 2019).

**DEMO data** (`USE_DEMO_DATA = True`): public SRH data found online, so the demo can answer more questions. **Not for the live platform**: set `USE_DEMO_DATA = False` (and `INTYOZA_DEMO=0` on the server) when Intyoza goes live.

| Demo source | What | Licence |
|---|---|---|
| HASH STI Q&A dataset (Infectious Diseases Institute, Makerere University, Uganda) | 2,502 questions from the public in 109 groups, answered by doctors | CC BY-NC 4.0 (non-commercial) |
| WHO fact sheets and Q&A | adolescent SRH, adolescent pregnancy, contraception, emergency contraception, STIs, HIV, cervical cancer, violence against women, abortion, infertility, adolescent mental health | CC BY-NC-SA 3.0 IGO |
| Menstrual Health Education bot data (GitHub) | about 545 short questions and answers on periods; not reviewed by health workers | none stated: lowest priority |

The answer model sees every demo item marked *DEMO SOURCE (not reviewed by Intyoza)*, below the approved answers, and is told never to use them for Rwandan laws, ages, services or phone numbers.

A failed download prints its reason; run the cell again, or download the file yourself into `demo_data/` (or `documents/`).

In [ ]:
import demo_data
importlib.reload(demo_data)
DOC_DIR, INDEX_DIR = os.path.join(FOLDER, "documents"), os.path.join(MODELS, "doc_index")
if USE_DEMO_DATA:
    demo_data.download(doc_dir=DOC_DIR)
    demo_data.build()
get_documents.prepare(bot.enc, DOC_DIR, INDEX_DIR)
docs = doc_engine.DocEngine(bot.enc, INDEX_DIR, use_llm=False, use_translator=False)
demo = demo_data.load() if USE_DEMO_DATA else None
kb = answer_engine.KnowledgeBase(bot, docs, DOC_DIR, demo=demo)
print(f"{len(bot.qa)} approved answers | {int(kb.doc_ok.sum())} document passages in use | {len(demo or [])} demo Q&A items")

## 4. Load the base model

AfriqueQwen3.5-4B-Instruct, downloaded the first time (about 9 GB), loaded in 4-bit.

In [ ]:
assert "bot" in globals() and "kb" in globals(), "Run steps 1, 2 and 3 first (Colab was restarted, so they must run again)."
gen = writer_training.load_model()
assert gen.ok, "The base model did not load (see the errors above)."
print(round(torch.cuda.memory_allocated() / 1e9, 1), "GB on the GPU")

## 5. Kinyarwanda questions for the demo data

The demo questions are in English. The answer model already reads English sources and answers in Kinyarwanda; to also learn how these questions are asked in Kinyarwanda, a sample is translated by the model (machine translation, demo only). About 15 minutes for 300; saved, so it is done only once.

In [ ]:
assert "kb" in globals() and "gen" in globals(), "Run steps 1 to 4 first (Colab was restarted, so they must run again)."
N_TRANSLATE = 0      # 0 = skip (the first runs kept no translations); try 50 to test it
if USE_DEMO_DATA and demo:
    n = demo_data.translate_questions(gen, demo, n=N_TRANSLATE, bot=bot)
    print(f"Translated {n} more; {sum(bool(it.get('questions_rw')) for it in demo)} demo items have a Kinyarwanda question")
    for it in [i for i in demo if i.get("questions_rw")][:8]:
        print(" ", it["q_en"], "->", it["questions_rw"][0])
    kb = answer_engine.KnowledgeBase(bot, docs, DOC_DIR, demo=demo)     # search now knows the Kinyarwanda questions

## 6. Training examples

Built from the training split only; the test examples come from the test split (and, for demo data, from demo items never used in training). One example of each task is printed.

In [ ]:
assert "kb" in globals(), "Run steps 1 to 5 first (Colab was restarted, so they must run again)."
train_ex = writer_training.build_examples(bot, docs, ex, "train", kb=kb)
test_ex = writer_training.build_examples(bot, docs, ex, "test", n_not_covered=10, n_check=24, n_understand=15,
                                         seed=1, kb=kb)
train_msgs = {e["message"] for e in train_ex}
print("train:", writer_training.summary_of(train_ex), "| from demo data:", sum(bool(e.get("demo_id")) or (e["qa_id"] is None and e["task"] != "understand") for e in train_ex))
print("test: ", writer_training.summary_of(test_ex))
print("test messages that are also in training:", sum(e["message"] in train_msgs for e in test_ex if e["qa_id"]))
assert any(e["task"] == "talk" for e in train_ex), ("No approved training conversations: set Review status to Approved in "
                                                   "the Conversations tab (INTYOZA_INCLUDE_DRAFTS=1 only for a test run).")
for task in ("talk", "crisis", "not_covered", "check", "understand"):
    e = next((x for x in train_ex if x["task"] == task), None)
    if e is None:
        continue
    print(f"\n========== {task} ==========\n{e['user'][-700:]}\n---> {e['target'][:400]}")

## 7. Base model on the test examples (before)

About 15 minutes. This is the starting point the fine-tuning must improve.

In [ ]:
before, before_table = writer_training.evaluate(gen, bot, test_ex, kb=kb)
pd.Series(before, name="base model").to_frame()

## 8. Fine-tuning

About 1 to 1.5 hours on a T4. The loss is printed every 10 steps and should go down. A checkpoint is saved on Drive every 50 steps; if Colab disconnects, run the notebook again from the top (you can skip steps 5 and 7) and it continues.

In [ ]:
OUT = os.path.join(MODELS, "intyoza_writer")
history, finished = writer_training.train(gen, train_ex, out_dir=OUT, epochs=1, lr=2e-4, rank=16, alpha=32,
                                          grad_accum=8, max_len=2560,
                                          time_limit_min=300)   # stops after 5 hours (Kaggle allows 12) and keeps what it learned
if history:
    import matplotlib.pyplot as plt
    h = pd.DataFrame(history)
    plt.figure(figsize=(7, 3)); plt.plot(h["step"], h["loss"]); plt.xlabel("step"); plt.ylabel("loss")
    plt.title("Fine-tuning loss"); plt.grid(alpha=0.3); plt.show()

## 9. Fine-tuned model on the test examples (after)

The same test examples as step 7.

| Result | Should be |
|---|---|
| talk: facts correct, ends with a question or offer, no stock opening | high |
| talk: similarity of replies to the same situation | lower than the base model: young people in the same situation get different words |
| crisis: gives the approved safety numbers exactly | 1.0 |
| not covered: correctly refused | high: the model says it doesn't know instead of guessing |
| check: catches the answer of a nearby question | high: this is what stops the "nearest answer" mistake |
| check: accepts the right answer | high: good answers are not thrown away |
| understand: search finds the question (understood) | higher than *message only* |

In [ ]:
after, after_table = writer_training.evaluate(gen, bot, test_ex, kb=kb)
compare = pd.DataFrame({"base model": before, "Intyoza fine-tuned": after})
compare

In [ ]:
# Replies written by the fine-tuned model (read them: correct, short unless asked, kind, natural Kinyarwanda?)
after_table[after_table["task"].isin(["talk", "crisis"])][["task", "lang", "message", "output", "expected"]].head(15)

In [ ]:
# Mistakes of the checker (each one is a wrong answer that would get through, or a good one thrown away)
after_table[(after_table["task"] == "check") & (after_table["correct"] == False)][["kind", "message", "output"]].head(15)

## 10. Save

Saves the results with the model, and a copy to download for the platform's server (`intyoza_writer.zip`: unzip it into `ai-service/models/`).

In [ ]:
writer_training.save(gen, OUT, train_ex, history, metrics={"before": before, "after": after})
compare.to_csv(os.path.join(MODELS, "intyoza_writer_results.csv"))
import shutil
shutil.make_archive(os.path.join(FOLDER, "intyoza_writer"), "zip", MODELS, "intyoza_writer")
print("Saved:", OUT, "and intyoza_writer.zip")
if IN_KAGGLE:      # Kaggle keeps only /kaggle/working: save the results there (small), for notebook 5 and the server
    keep = "/kaggle/working/intyoza_outputs"
    for part in ("models/intyoza_writer", "models/doc_index", "demo_data"):
        if os.path.exists(part):
            shutil.copytree(part, os.path.join(keep, part), dirs_exist_ok=True)
    for f in glob.glob("documents/*") + ["models/intyoza_writer_results.csv", "intyoza_writer.zip"]:
        if os.path.isfile(f) and (f.startswith("documents/DEMO_") or f.startswith("documents/Rwanda_") or not f.startswith("documents/")):
            os.makedirs(os.path.join(keep, os.path.dirname(f)), exist_ok=True)
            shutil.copy(f, os.path.join(keep, f))
    print("Kaggle: results saved in the Output tab (intyoza_outputs). Add this notebook's output as input to notebook 5.")
print("Next: 5_Chatbot_Colab.ipynb")